# OrbitalThrusterEnv — RL Training Notebook (SFT → GRPO)

Trains a small open LLM to act as a long-horizon mission-ops controller inside `OrbitalThrusterEnv`.

**Stack**: Unsloth + TRL (`SFTTrainer` → `GRPOTrainer`) + OpenEnv verifier rewards.

**Default model**: `Qwen/Qwen2.5-3B-Instruct` (fits 4060 laptop, 8 GB VRAM, 4-bit). Override via `ORBITAL_BASE_MODEL` env var to `Qwen/Qwen3-4B-Instruct-2507` when running on Hugging Face credits.

**Pipeline**: seed trajectories → SFT (format priming) → GRPO with 5 independent reward funcs → eval vs baselines → plots.

**Anti-reward-hacking**: format check, env-step verifier, recommended-mode check, action-spam penalty, fuel-discipline penalty (5 independent signals).

## 1. Environment setup

In [ ]:
import os, sys
from pathlib import Path
ROOT = Path.cwd().parent if Path.cwd().name == 'training' else Path.cwd()
if str(ROOT) not in sys.path: sys.path.insert(0, str(ROOT))
if str(ROOT / 'training') not in sys.path: sys.path.insert(0, str(ROOT / 'training'))
os.environ.setdefault('ORBITAL_BASE_MODEL', 'Qwen/Qwen2.5-3B-Instruct')
print('ROOT:', ROOT)
print('MODEL:', os.environ['ORBITAL_BASE_MODEL'])

In [ ]:
# One-time install (uncomment when running fresh)
# !pip install -q -r training/requirements.txt

## 2. Generate / load seed trajectories from the tuned-PD expert

In [ ]:
from common import collect_seed_records
seed_path = ROOT / 'training' / 'data' / 'seed_trajectories.jsonl'
if not seed_path.exists():
    collect_seed_records(seed_path, episodes_per_task=3, max_records_per_task=128)
print('seed file size:', seed_path.stat().st_size, 'bytes')

## 3. Baseline rollout (random / deterministic / tuned-PD)

In [ ]:
import subprocess
subprocess.run([sys.executable, str(ROOT / 'training' / 'evaluate_baselines.py')], check=True)

## 4. SFT — short JSON-format priming (~80 steps QLoRA)

In [ ]:
from qwen3_smoke_sft import main as run_sft
run_sft()

## 5. GRPO — verifier-driven RL with multi-component rewards

Reward components (each independent, summed by GRPO):
- `reward_format` — JSON parses + valid enums + has reason
- `reward_env_step` — replays history, scores candidate via real env
- `reward_mode_match` — control_mode ∈ recommended for current directive
- `reward_anti_spam` — penalty if action repeated ≥ 4× in last 6 steps
- `reward_fuel_discipline` — bonus low-fuel→idle, penalty low-fuel→large pulse

In [ ]:
from qwen3_grpo_train import main as run_grpo
run_grpo()

## 6. Reward + loss curves

In [ ]:
from rl_utils import TRAIN_LOG_DIR, plot_training_curves
plot_training_curves(TRAIN_LOG_DIR / 'grpo_metrics.csv', TRAIN_LOG_DIR / 'grpo_metrics.png')
from IPython.display import Image, display
png = TRAIN_LOG_DIR / 'grpo_metrics.png'
if png.exists(): display(Image(str(png)))

## 7. Eval — baseline vs trained on all 4 tasks

In [ ]:
subprocess.run([sys.executable, str(ROOT / 'training' / 'eval_trained_model.py')], check=True)

In [ ]:
from IPython.display import Image, display
import pandas as pd
out_dir = ROOT / 'outputs' / 'eval_trained'
df = pd.read_csv(out_dir / 'trained_vs_baseline.csv')
display(df)
display(Image(str(out_dir / 'trained_vs_baseline.png')))

## 8. Anti-reward-hacking sanity checks

In [ ]:
import json
from collections import Counter
from common import rollout_task, TASK_IDS
from rl_utils import GRPO_OUTPUT_DIR, DEFAULT_MODEL, make_lora_controller
ctrl = make_lora_controller(GRPO_OUTPUT_DIR, base_model=DEFAULT_MODEL)
task = 'mission_ops_long_horizon'
trace = rollout_task(task, ctrl, record_history=True)
actions = [t['expert_action']['action_type'] for t in trace['trace']]
counts = Counter(actions)
idle_frac = counts.get('idle', 0) / max(len(actions), 1)
top_action, top_count = counts.most_common(1)[0]
print(f'task={task} success={trace["success"]} reward_total={trace["reward_total"]:.3f}')
print(f'fuel_used={trace["fuel_used"]:.2f} milestones={trace["milestones_completed_count"]}')
print(f'action_diversity_top={top_action}({top_count}) idle_fraction={idle_frac:.2%}')
assert idle_frac < 0.85, 'all-idle exploit detected'
assert top_count / len(actions) < 0.85, 'single-action exploit detected'
out = ROOT / 'outputs' / 'training' / 'sample_rollout_flagship.json'
out.write_text(json.dumps({k: trace[k] for k in ['task_id','success','reward_total','fuel_used','milestones_completed_count','reward_columns']}, indent=2))
print('saved:', out)

## 9. Save adapter (LoRA only — never naive 4-bit→16-bit merge)

GRPO trainer already saved to `trainer_output/qwen_grpo/`. To push to HF Hub:

```python
from huggingface_hub import HfApi
HfApi().upload_folder(folder_path='trainer_output/qwen_grpo', repo_id='<user>/orbital-thruster-grpo', repo_type='model')
```